# Core Idea: How to Access History Token
Question: For each token at position `t`, how to look *back* and summarize all past tokens  

"Bag of words" (`bow`)
* Simplest answer: take the average of all previous tokens
* Loses **order** information

Self-attention
* Data dependent weights

## 1 - Bag of Words
Take the average of previous tokens in three ways
1. `for` loop
2. Matrix multiplication (efficient)
3. `softmax` function (prepare for attention)

In [2]:
import torch

# Test tensor
B, T, C = 4, 8, 2

x = torch.randn(B, T, C)
xbow = torch.zeros((B, T, C)) 


### Version 1: `for` Loop
Nested Python loops: slow, can't go parallel on GPU

In [3]:
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]                      # (t+1, C), all tokens up to t
        xbow[b, t] = torch.mean(xprev, 0)       # (t+1, C) -> (C, ) average over time dimension

### Version 2: Matrix Multiplication
Efficient for "weighted aggregation"

Weighted aggregation: combine a set of items by assigning a "weight" to each one then take the sum  

$
\begin{aligned}
\text{Result} = \sum_i \omega_i \cdot \mathbf{v}_i
\end{aligned}
$

In [4]:
# Toy model using matrix multiplication for a "weighted aggregation":
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

a = torch.tril(torch.ones(3, 3))            # lower triangular matrix
print('a=')                                 # pos `t` only look at pos `<= t`
print(a)

factor = a.sum(dim=1, keepdim=True)         # sum along dimension 1 (rows)
print('\nDivide each element by the sum of its row:')
print(factor)

a = a / factor                              # normalize each row to sum to 1
print('\nNormalized a=')
print(a)

# Test tensor
b = torch.randint(low=0, high=10, size=(3, 2), dtype=torch.float32)
print('\nb=')
print(b)

# Weighted aggregation 
# Each row of `c` is a weighted average of rows of `b`
# Weights come from `a`
c = a @ b                                   # (X, X) @ (X, Y) -> (X, Y)
print('\nc=')
print(c)


a=
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

Divide each element by the sum of its row:
tensor([[1.],
        [2.],
        [3.]])

Normalized a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])

c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


Weighted matrix $C_{(X, Y)} = A_{(X, X)} \cdot B_{(X, Y)}$, where 
$c_{ij} = \sum_k^{n=X} a_{ik}b_{kj}$

In [5]:
# Weighted aggregation
wei = torch.tril(torch.ones(T, T))           # (T, T) mask filtering out future tokens
wei = wei / wei.sum(1, keepdim=True)         # averaging weights
print(wei.shape)

# For Reference
# B, T, C = 4, 8, 2
# x = torch.randn(B, T, C)
print(x.shape)

# (T, T) @ (T, C) -> (T, C) for each batch
# (T, T) -> padded to (1, T, T) -> broadcast to (B, T, T)
# So (T, T) @ (B, T, C) -> (B, T, C)
xbow2 = wei @ x 
print(xbow2.shape)          

# Check xbow2
torch.allclose(xbow, xbow2)

torch.Size([8, 8])
torch.Size([4, 8, 2])
torch.Size([4, 8, 2])


True

### Version 3: SoftMax
`wei`: How much do we want each past token to aggregate to the future

Masking: Token from the past can't communicate with the future by setting them to negative inifity


Same effect as version 2 but used in real attention
* Here `wei` start with 0
* In attention `wei` are query-key dot products

In [6]:
tril = torch.tril(torch.ones(3, 3))
a = torch.zeros((3, 3))
a = a.masked_fill(tril == 0, float('-inf'))     # upper triangle (future tokens) -> -inf
print(a)

from torch.nn import functional as F
a = F.softmax(a, dim=-1)                        # softmax each row `exp(-inf) = 0`
print(a)                                        # same effect as `tril / tril.sum`

tensor([[0., -inf, -inf],
        [0., 0., -inf],
        [0., 0., 0.]])
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])


In [7]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))   # upper triangle (future tokens) -> -inf

from torch.nn import functional as F
wei = F.softmax(wei, dim=-1)                      # softmax each row

print(wei.shape)
print(x.shape)

xbow3 = wei @ x 
print(xbow3.shape)

# Check xbow3
torch.allclose(xbow, xbow3)

torch.Size([8, 8])
torch.Size([4, 8, 2])
torch.Size([4, 8, 2])


True

# 2 - Self-Attention

Weights depend on data:
* Each token "decides" how much to attend to previous tokens

Query, Key, Value: similar to a dictionary lookup
* Query: what to look for
* Key: what does it have
* Value: what to retrieve

Have a **query**, compare it to all the **keys**, then retrieve the corresponding **value**

But a rather "soft" lookup
* `dict`: a key either matches or doesn't
* Attention: every key matches a little, weighted by similarity

"Self"-attention: keys and values are from the same source as queries

---
## Single Head Attention
For one head $i$:

Input: $X \in \mathbb{R}^{L \times d_{\text{model}}}$ -> `x` - `(T, C)`
* A sequence of $L$ tokens, each represented by a vector of $d_\text{model}$ dimensions

### 2-1 Linear Projections

For $h$ heads, query and key have dimension $d_k$, value have dimension $d_v$, where
$d_k = d_v = d_{\text{model}} / h$

* $Q_i = X \, W_Q^{(i)} \quad$ where 
$W_Q^{(i)} \in \mathbb{R}^{d_{\text{model}} \times d_k}$ 
->
$Q_i \in \mathbb{R}^{L \times d_k}$
-> `q` - `(T, head_size)`

* $K_i = X \, W_K^{(i)} \quad$ where 
$W_K^{(i)} \in \mathbb{R}^{d_{\text{model}} \times d_k}$
->
$K_i \in \mathbb{R}^{L \times d_k}$
-> `k` - `(T, head_size)`

* $V_i = X \, W_V^{(i)} \quad$ where 
$W_V^{(i)} \in \mathbb{R}^{d_{\text{model}} \times d_v}$
->
$V_i \in \mathbb{R}^{L \times d_v}$
-> `v` - `(T, head_size)`


In [8]:
import torch.nn as nn

torch.manual_seed(RANDOM_SEED)

B, T, C = 4, 8, 32                 # Each batch processed independently
x = torch.randn((B, T, C))
print('x  ', x.shape)

# Single Head self-attention
head_size = 16

# weight matrix (head_size, C)
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
print('wk   ', key.weight.shape, ' wq  ', query.weight.shape, ' wv  ', value.weight.shape)
print('wk.T ', key.weight.transpose(-2, -1).shape, 
        ' wq.T', query.weight.transpose(-2, -1).shape,
        ' wv.T', value.weight.transpose(-2, -1).shape)

print('\nkey = x @ wk.T,             query = x @ wq.T')

k = key(x)                        # (B, T, C) @ (C, head_size) -> (B, T, head_size) 
q = query(x)
print('k  ', k.shape, ' q   ', q.shape)

x   torch.Size([4, 8, 32])
wk    torch.Size([16, 32])  wq   torch.Size([16, 32])  wv   torch.Size([16, 32])
wk.T  torch.Size([32, 16])  wq.T torch.Size([32, 16])  wv.T torch.Size([32, 16])

key = x @ wk.T,             query = x @ wq.T
k   torch.Size([4, 8, 16])  q    torch.Size([4, 8, 16])


### 2-2 Scaled dot-product attention scores

Each element $(\alpha, \beta)$ in the $\text{Scores}_i$ matrix represents the similarity between **query token $\alpha$** and **key token $\beta$**

* $\text{Scores}_i = \frac{Q_i K_i^\top}{\sqrt{d_k}} \quad \in \mathbb{R}^{L \times L}$
-> `wei` - `(T, T)`

* Need to normalize with $\sqrt{d_k}$ (`head_size`): dot products grow large in magnitude and push softmax functions to extremely small gradients

In [9]:
print('q  ', q.shape, '   k.T', k.transpose(-2, -1).shape)

# Attention weights
wei = q @ k.transpose(-2, -1)     # (B, T, head_size) @ (B, head_size, T) -> (B, T, T) 
print('\nAttention weights = query @ key.T')
print('wei', wei.shape)

# Normalize with `head_size**-0.5` (\sqrt{d_k})
wei = wei * head_size**-0.5


q   torch.Size([4, 8, 16])    k.T torch.Size([4, 16, 8])

Attention weights = query @ key.T
wei torch.Size([4, 8, 8])



---
For independent random variables with mean `0` and variance `1`, dot product $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ has mean `0` and variance $d_k$. 

Scaling by $\sqrt{d_k}$ (`head_size`) to reduce variance to `1`


In [15]:
# An illustration
k_test = torch.randn(B, T, head_size)
q_test = torch.randn(B, T, head_size)
wei_test = q_test @ k_test.transpose(-2, -1)

# `.var(correction=0)` to divide by N rather than N-1
print(f'Random k: variance {k_test.var(correction=0).item():.4f}  | mean {k_test.mean().item():.4f}')    
print(f'Random q: variance {q_test.var(correction=0).item():.4f}  | mean {q_test.mean().item():.4f}')

print('d_k     :', head_size)

print('\nDot product before normalization: ')
print(f'          variance {wei_test.var(correction=0).item():.4f} | mean {wei_test.mean().item():.4f}')

wei_test = wei_test * head_size**-0.5
print('Dot product after normalization: ')
print(f'          variance {wei_test.var(correction=0).item():.4f}  | mean {wei_test.mean().item():.4f}')

Random k: variance 1.0326  | mean -0.0528
Random q: variance 0.9600  | mean 0.0207
d_k     : 16

Dot product before normalization: 
          variance 15.0538 | mean 0.4030
Dot product after normalization: 
          variance 0.9409  | mean 0.1008


### 2-3 Masking and SoftMax
#### Masking

Decoder attention block: masking out future tokens with lower triangular matrix (what GPT uses)

Encode attention block: does not mask out future tokens, so that all tokens communicate


#### SoftMax

Apply softmax per row along the last dimension, over the keys. Converts scores to probabilities

* $A_i = \text{softmax}(\text{Scores}_i) \quad \in \mathbb{R}^{L \times L}$
-> `wei` - `(T, T)`

Now each row $A_i[\alpha, :]$ sums to $1$. It's a distribution of attention weights from **query token $\alpha$** to all tokens

In [11]:
# Masking and SoftMax
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril==0, float('-inf'))  # block future tokens for a GPT decoder
wei = F.softmax(wei, dim=-1)                   # normalize to probabilities

print('\nProbability matrix:')
print('wei', wei[0].detach())


Probability matrix:
wei tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4106, 0.5894, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3657, 0.2283, 0.4061, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2168, 0.2759, 0.2204, 0.2870, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2553, 0.1697, 0.1548, 0.2341, 0.1861, 0.0000, 0.0000, 0.0000],
        [0.1318, 0.2060, 0.1405, 0.1917, 0.1949, 0.1351, 0.0000, 0.0000],
        [0.2137, 0.0978, 0.2374, 0.1025, 0.1418, 0.0838, 0.1230, 0.0000],
        [0.0852, 0.1047, 0.0824, 0.1376, 0.1015, 0.1900, 0.1780, 0.1206]])


### 2-4 Weighted sum of values

Use the weights to aggregate the value vectors:

* $\text{head}_i = A_i \, V_i \quad \in \mathbb{R}^{L \times d_v}$
-> `out` - `(T, head_size)`

This is the output of head $i$, a weighted combination of the **value** vectors of all **keys**, for each **query** position

In [12]:
# Weighted value
v = value(x)                      # (B, T, C) @ (C, head_size) -> (B, T, head_size) 
print('\nWeighted values = wei @ value')
print('v  ', v.shape)

out = wei @ v                     # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
print('out', out.shape)


Weighted values = wei @ value
v   torch.Size([4, 8, 16])
out torch.Size([4, 8, 16])


## Self-Attention Summary

In [13]:
# The complete process
torch.manual_seed(RANDOM_SEED)

B, T, C = 4, 8, 32                 # Each batch processed independently
x = torch.randn((B, T, C))
print('x  ', x.shape)

# Single Head self-attention
head_size = 16

# weight matrix (head_size, C)
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
print('wk   ', key.weight.shape, '   wq  ', query.weight.shape)
print('wk.T ', key.weight.transpose(-2, -1).shape, 
        '   wq.T', query.weight.transpose(-2, -1).shape)


print('\nkey = x @ wk.T, query = x @ wq.T')

k = key(x)                        # (B, T, C) @ (C, head_size) -> (B, T, head_size) 
q = query(x)
print('k  ', k.shape, '   q ', q.shape)
print('k.T', k.transpose(-2, -1).shape)

# Attention weights
wei = q @ k.transpose(-2, -1)     # (B, T, head_size) @ (B, head_size, T) -> (B, T, T) 
print('\nAttention weights = query @ key.T')
print('wei', wei.shape)

# Normalize with `head_size**-0.5` (\sqrt{d_k})
wei = wei * head_size**-0.5

# Masking and SoftMax
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril==0, float('-inf'))  # block future tokens for a GPT decoder
wei = F.softmax(wei, dim=-1)                   # normalize to probabilities

# Weighted value
v = value(x)                      # (B, T, C) @ (C, head_size) -> (B, T, head_size) 
print('\nWeighted values = wei @ value')
print('v  ', v.shape)

out = wei @ v                     # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
print('out', out.shape)

print('\nProbability matrix:')
print('wei', wei[0].detach())

print('\nWeighted values:')
print('out', out[0].detach())


x   torch.Size([4, 8, 32])
wk    torch.Size([16, 32])    wq   torch.Size([16, 32])
wk.T  torch.Size([32, 16])    wq.T torch.Size([32, 16])

key = x @ wk.T, query = x @ wq.T
k   torch.Size([4, 8, 16])    q  torch.Size([4, 8, 16])
k.T torch.Size([4, 16, 8])

Attention weights = query @ key.T
wei torch.Size([4, 8, 8])

Weighted values = wei @ value
v   torch.Size([4, 8, 16])
out torch.Size([4, 8, 16])

Probability matrix:
wei tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4106, 0.5894, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3657, 0.2283, 0.4061, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2168, 0.2759, 0.2204, 0.2870, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2553, 0.1697, 0.1548, 0.2341, 0.1861, 0.0000, 0.0000, 0.0000],
        [0.1318, 0.2060, 0.1405, 0.1917, 0.1949, 0.1351, 0.0000, 0.0000],
        [0.2137, 0.0978, 0.2374, 0.1025, 0.1418, 0.0838, 0.1230, 0.0000],
        [0.0852, 0.1047, 0.0824, 0.1376, 0.1015, 0.1900

# Multi-head Attention Preview

Now each $\text{head}_i$ has shape $(L, d_v)$ for $i = 1, ..., h$ -> `out` - `(T, head_size)`

Single-head vs. Multi-head:
* A single attention head tends to pay attention to a **limited kind of relationship** between tokens at a time (topic, syntax, punctuations, etc.)
* Multiple heads run in **parallel**, each potentially learning different relationship types

We run $h$ single heads in parallel, then concatenate them along the feature dimension (the last dimension)

* $\text{MultiHeadConcatenated} = [\text{head}_1 ; \text{head}_2 ; \dots ; \text{head}_h] \quad \in  \mathbb{R}^{L \times (h \cdot d_v)}$

For $d_v = d_{\text{model}} / h$ (`head_size = C // h`)
* $h \cdot d_v = d_{\text{model}}$
* The concatenated shape is back to $(L, d_{\text{model}})$

After concatenating, one more linear layer $W_O$ mixes information across heads
* $\text{MultiHead} = [\text{head}_1; \dots; \text{head}_h]\, W_O\quad$, 
where $W_O \in \mathbb{R}^{d{\text{model}} \times d{\text{model}}}$ -> shape stays $(L, d_{\text{model}})$
* Each head computed independently, so this lets them talk

Tracing shape changes:
* Single head:   `x` `(T, C)` -> `out` `(T, head_size)`
* $h$ heads:       `x` `(T, C)` -> `h` × `(T, head_size)` -concat-> `(T, h·head_size = C)` -wo-> `(T, C)`
